# Machine Learning Practical Assignment
## Data Preprocessing & Feature Selection
### Adult Census Income Dataset

**Group Members**

| Student | Roll Number |
|---|---:|
| Sachin Kumar | 037 |
| Rohit Singh | 036 |
| Shubham Kumar Gupta | 043 |
| Jay Singh | 018 |

### Project Philosophy
**Understand → Calculate → Code → Verify → Interpret**

This notebook is designed around the requirements of the supplied practical-assignment PDF.

# 1. Problem Definition

### Real-world problem
The Adult Census Income dataset represents a classification problem in which demographic, educational, occupational and financial characteristics are used to study whether an individual's annual income falls above or below a specified income threshold.

### Objective
Our objective is to:
- understand and clean the dataset,
- implement important preprocessing techniques from scratch,
- compare manual calculations with library implementations,
- study relationships between features and the target,
- select useful features while avoiding data leakage.

### Target
`income`

### Problem Type
**Binary Classification**

# 2. Dataset Understanding

The Adult/Census Income dataset contains demographic, educational, occupational and financial variables.

The raw UCI version uses the following fields:

`age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country, income`

The target is `income`.

> **Important:** We will inspect the raw data before deciding which columns to remove. No feature will be removed merely because it is inconvenient.

In [ ]:
import os, sys, math, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")

from preprocessing import *
from feature_selection import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

# 3. Load the Dataset

The notebook can download the UCI Adult data automatically.

The data files are downloaded from the UCI Machine Learning Repository. If `adult.csv` is already placed in `dataset/`, that local file will be used instead.

In [ ]:
DATA_DIR = "../dataset"
os.makedirs(DATA_DIR, exist_ok=True)

csv_path = os.path.join(DATA_DIR, "adult.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"
    ]
    raw = pd.read_csv(
        url, header=None, names=columns, na_values=[" ?", "?"],
        skipinitialspace=True
    )
    raw.to_csv(csv_path, index=False)
    df = raw.copy()

print("Shape:", df.shape)
df.head()

# 4. Initial Data Exploration

We inspect:
- first records,
- last records,
- shape,
- column names,
- data types,
- unique values,
- missing values,
- descriptive statistics.

In [ ]:
print("First five records:")
display(df.head())

print("Last five records:")
display(df.tail())

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nUnique values:")
display(df.nunique().to_frame("unique_values"))

print("\nMissing values:")
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": df.isna().mean()*100
})
display(missing)

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

# 5. Basic Statistics — From Scratch

For a numerical feature, we calculate mean, median, mode, variance and standard deviation manually and then verify the results with Pandas.

The population variance used for this assignment is:

\[
Var(X)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2
\]

In [ ]:
numeric_demo = "age"
values = df[numeric_demo].dropna()

manual_stats = {
    "Mean": manual_mean(values),
    "Median": manual_median(values),
    "Mode": manual_mode(values),
    "Variance": manual_variance(values),
    "Standard Deviation": manual_std(values),
    "Minimum": values.min(),
    "Maximum": values.max(),
    "Range": values.max()-values.min()
}
display(pd.Series(manual_stats, name="Manual"))

verification = {
    "Mean": values.mean(),
    "Median": values.median(),
    "Mode": values.mode().iloc[0],
    "Variance": values.var(ddof=0),
    "Standard Deviation": values.std(ddof=0),
    "Minimum": values.min(),
    "Maximum": values.max(),
    "Range": values.max()-values.min()
}
display(pd.Series(verification, name="Pandas"))

print("Interpretation: the manual and library results should agree up to normal floating-point precision.")

# 6. Missing Value Analysis and Treatment

We first quantify missing values. For the primary implementation we use basic Python/Pandas logic instead of `SimpleImputer`.

Decision rule:
- numerical variables: inspect distribution; use median when outliers/skewness make it more robust,
- categorical variables: use mode when the missing proportion is small,
- a feature with excessive missingness may be removed only with a clear justification.

In [ ]:
before_missing = df.isna().sum()

# Work on a copy
clean_df = df.copy()

# Manual-style treatment
for col in clean_df.columns:
    if clean_df[col].isna().sum() == 0:
        continue
    if pd.api.types.is_numeric_dtype(clean_df[col]):
        fill_value = manual_median(clean_df[col].dropna())
    else:
        fill_value = manual_mode(clean_df[col].dropna())
    clean_df[col] = clean_df[col].fillna(fill_value)

after_missing = clean_df.isna().sum()

comparison = pd.DataFrame({
    "Before Missing": before_missing,
    "After Missing": after_missing
})
display(comparison)

print("Interpretation: missing values are treated using rules based on variable type and distribution rather than blindly deleting records.")

# 7. Duplicate and Invalid Data

We inspect duplicate observations and basic logical inconsistencies.

For this dataset, examples include:
- negative age,
- negative hours worked,
- impossible values in count-like variables,
- inconsistent whitespace/capitalization in categorical fields.

In [ ]:
duplicate_count = clean_df.duplicated().sum()
print("Original records:", len(clean_df))
print("Duplicate records:", duplicate_count)

clean_df = clean_df.drop_duplicates().copy()

# Basic validity checks
invalid_age = (clean_df["age"] <= 0).sum()
invalid_hours = (clean_df["hours-per-week"] < 0).sum()
invalid_capital_gain = (clean_df["capital-gain"] < 0).sum()
invalid_capital_loss = (clean_df["capital-loss"] < 0).sum()

print("\nInvalid-value checks:")
print("Age <= 0:", invalid_age)
print("Hours-per-week < 0:", invalid_hours)
print("Capital-gain < 0:", invalid_capital_gain)
print("Capital-loss < 0:", invalid_capital_loss)

# Normalize categorical whitespace/case
categorical_cols = clean_df.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    clean_df[col] = clean_category(clean_df[col])

print("\nRecords after duplicate treatment:", len(clean_df))

# 8. Categorical Encoding — From Scratch

### Label Encoding
Label encoding maps categories to integer codes. It is suitable when the categories have a meaningful order or when the encoded representation is being used carefully.

### One-Hot Encoding
One-hot encoding creates a separate binary column for each category and avoids imposing an artificial numerical order on nominal categories.

We implement both approaches with dictionaries/basic logic first.

In [ ]:
# Label encoding demonstration on sex
label_encoded_sex, sex_mapping = manual_label_encode(clean_df["sex"])
print("Label mapping:", sex_mapping)
display(pd.DataFrame({"sex": clean_df["sex"].head(10), "encoded": label_encoded_sex.head(10)}))

# One-hot demonstration on workclass
one_hot_demo = manual_one_hot(clean_df["workclass"])
display(one_hot_demo.head())

# 9. Outlier Detection — IQR

For a numerical feature:

\[
IQR=Q3-Q1
\]

\[
Lower=Q1-1.5(IQR)
\]

\[
Upper=Q3+1.5(IQR)
\]

We will not automatically delete outliers. We first determine whether they are errors or valid extreme observations.

In [ ]:
outlier_feature = "capital-gain"
q1, q3, iqr, lower, upper = manual_iqr_bounds(clean_df[outlier_feature])

iqr_outliers = clean_df[(clean_df[outlier_feature] < lower) | (clean_df[outlier_feature] > upper)]

print("Feature:", outlier_feature)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Detected outliers:", len(iqr_outliers))

# 10. Outlier Detection — Z-Score

\[
z=\frac{x-\mu}{\sigma}
\]

A common rule for potential extreme observations is:

\[
|z|>3
\]

IQR and Z-score can produce different results because IQR is based on quartiles and is more robust to extreme values, while the Z-score depends directly on mean and standard deviation.

In [ ]:
z_values = manual_z_scores(clean_df[outlier_feature].astype(float))
z_outliers = clean_df[np.abs(z_values) > 3]

print("Z-score outliers:", len(z_outliers))
print("IQR outliers:", len(iqr_outliers))

# We retain valid extremes rather than blindly deleting them.
print("Decision: inspect domain validity before removing mathematical outliers.")

# 11. Data Transformation

We investigate skewness and use a log transformation where appropriate.

For non-negative monetary/count variables, `log1p(x) = log(1+x)` is safer than `log(x)` when zeros exist.

We compare the distribution before and after transformation.

In [ ]:
transform_feature = "capital-gain"

fig = plt.figure(figsize=(8,5))
plt.hist(clean_df[transform_feature], bins=40)
plt.title(f"Distribution Before Log Transformation: {transform_feature}")
plt.xlabel(transform_feature)
plt.ylabel("Frequency")
plt.show()

clean_df[transform_feature + "_log"] = np.log1p(clean_df[transform_feature])

fig = plt.figure(figsize=(8,5))
plt.hist(clean_df[transform_feature + "_log"], bins=40)
plt.title(f"Distribution After Log Transformation: {transform_feature}")
plt.xlabel("log1p(" + transform_feature + ")")
plt.ylabel("Frequency")
plt.show()

print("Skewness before:", clean_df[transform_feature].skew())
print("Skewness after:", clean_df[transform_feature + "_log"].skew())

# 12. Feature Scaling — Min-Max Normalization

\[
X' = \frac{X-X_{min}}{X_{max}-X_{min}}
\]

The primary implementation is performed manually. We then verify the result with a library implementation.

In [ ]:
scale_feature = "age"
age_values = clean_df[scale_feature].astype(float).tolist()

manual_norm = manual_minmax(age_values)
norm_series = pd.Series(manual_norm, index=clean_df.index, name="manual_normalized_age")

display(pd.DataFrame({
    "Original": clean_df[scale_feature].head(10),
    "Min": clean_df[scale_feature].min(),
    "Max": clean_df[scale_feature].max(),
    "Normalized": norm_series.head(10)
}))

print("Minimum normalized value:", min(manual_norm))
print("Maximum normalized value:", max(manual_norm))

# 13. Feature Scaling — Standardization

\[
Z=\frac{X-\mu}{\sigma}
\]

The mean and standard deviation are calculated manually.

**Data-leakage rule:** when building the final ML pipeline, these parameters must be learned from the training set and then applied to the test set.

In [ ]:
standardized, train_mean_demo, train_std_demo = manual_standardize(age_values)
display(pd.DataFrame({
    "Original": clean_df[scale_feature].head(10),
    "Standardized": pd.Series(standardized, index=clean_df.index).head(10)
}))

print("Manual mean:", train_mean_demo)
print("Manual standard deviation:", train_std_demo)

# 14. Data Visualization

Required meaningful visualizations:
- histogram,
- box plot,
- bar chart,
- scatter plot,
- correlation heatmap.

Every graph should have a title, axis labels and an interpretation.

In [ ]:
# Histogram
fig = plt.figure(figsize=(8,5))
plt.hist(clean_df["age"], bins=30)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

# Box plot
fig = plt.figure(figsize=(8,5))
plt.boxplot(clean_df["hours-per-week"])
plt.title("Hours Per Week — Box Plot")
plt.ylabel("Hours Per Week")
plt.show()

# Bar chart
income_counts = clean_df["income"].value_counts()
fig = plt.figure(figsize=(7,5))
plt.bar(income_counts.index.astype(str), income_counts.values)
plt.title("Income Class Distribution")
plt.xlabel("Income Class")
plt.ylabel("Number of Records")
plt.show()

# Scatter plot
fig = plt.figure(figsize=(8,5))
plt.scatter(clean_df["age"], clean_df["hours-per-week"], alpha=0.2)
plt.title("Age vs Hours Per Week")
plt.xlabel("Age")
plt.ylabel("Hours Per Week")
plt.show()

# Correlation heatmap using Matplotlib
numeric_df = clean_df.select_dtypes(include=np.number)
corr = numeric_df.corr()
fig = plt.figure(figsize=(9,7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Numerical Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# 15. Train-Test Split — From Scratch

We randomly shuffle row indices and divide them into training and testing sets.

An 80/20 split is used.

The same observations must not be used both for training and final evaluation.

In [ ]:
# Create the final modelling dataset and split BEFORE learning preprocessing parameters.
# This is the leakage-free pipeline used for the final feature-selection analysis.
model_df = df.copy()

# Remove exact duplicate rows before splitting (unsupervised cleaning).
model_df = model_df.drop_duplicates().copy()

# Normalize categorical text without using test information.
for col in model_df.select_dtypes(include="object").columns:
    model_df[col] = clean_category(model_df[col])

# Create binary target.
model_df["target"] = (model_df["income"].str.contains(">50k")).astype(int)
model_df = model_df.drop(columns=["income"])

# Manual 80/20 split.
train_df, test_df = manual_train_test_split(model_df, test_size=0.20, seed=42)

# Learn missing-value parameters ONLY from training data, then apply them to both sets.
train_df = train_df.copy()
test_df = test_df.copy()

for col in train_df.columns:
    if col == "target":
        continue
    if train_df[col].isna().sum() == 0:
        continue
    if pd.api.types.is_numeric_dtype(train_df[col]):
        fill_value = manual_median(train_df[col].dropna())
    else:
        fill_value = manual_mode(train_df[col].dropna())
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col] = test_df[col].fillna(fill_value)

print("Leakage-free final pipeline")
print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Training proportion:", len(train_df)/len(model_df))
print("Testing proportion:", len(test_df)/len(model_df))
print("Remaining train missing values:", int(train_df.isna().sum().sum()))
print("Remaining test missing values:", int(test_df.isna().sum().sum()))
print("All learned imputation parameters came from TRAINING data only.")


# 16. Data Leakage Demonstration

### Incorrect
Entire dataset → calculate preprocessing parameters → transform → split

This allows information from the test set to influence preprocessing.

### Preferred
Dataset → split → learn parameters from training set → transform training set → transform test set using the same training parameters.

The same principle applies to:
- imputation,
- normalization,
- standardization,
- supervised feature selection,
- category mappings where learned from data.

In [ ]:
# Demonstration with age
train_age = train_df["age"].astype(float)
test_age = test_df["age"].astype(float)

mu_train = manual_mean(train_age)
sd_train = manual_std(train_age)

train_z = (train_age - mu_train) / sd_train
test_z = (test_age - mu_train) / sd_train

print("Training mean:", mu_train)
print("Training SD:", sd_train)
print("Test transformation uses TRAINING mean and SD.")
display(pd.DataFrame({
    "train_example": train_z.head().values
}))

# 17. Preprocessing Pipeline

Our logical sequence is:

**Raw Dataset**
→ Understand Dataset
→ Initial Exploration
→ Train-Test Split
→ Learn Training Parameters
→ Missing Value Treatment
→ Duplicate/Invalid Data Treatment
→ Encoding
→ Outlier Analysis/Treatment
→ Transformation
→ Scaling
→ Feature Selection
→ Final ML-Ready Dataset

The key reason for this sequence is to prevent information from the test set leaking into learned preprocessing or supervised feature-selection decisions.

# 18. Feature Selection — Variance

A zero-variance feature has the same value for every observation and therefore provides no discriminatory variation.

We calculate variance manually and inspect low-variance numerical variables.

In [ ]:
variance_results = []
for col in train_df.select_dtypes(include=np.number).columns:
    if col == "target":
        continue
    variance_results.append([col, manual_variance(train_df[col])])

variance_table = pd.DataFrame(variance_results, columns=["Feature","Manual Variance"])
display(variance_table.sort_values("Manual Variance"))
print("Variance was calculated using training data only.")


# 19. Feature Selection — Pearson Correlation

Pearson correlation measures the strength and direction of a **linear** relationship.

A low Pearson correlation does not prove that no relationship exists; nonlinear relationships may still be present.

In [ ]:
pearson_results = []
for col in train_df.select_dtypes(include=np.number).columns:
    if col == "target":
        continue
    r = manual_pearson(train_df[col], train_df["target"])
    pearson_results.append([col, r])

pearson_table = pd.DataFrame(pearson_results, columns=["Feature","Pearson r"])
display(pearson_table.sort_values("Pearson r", key=lambda s: s.abs(), ascending=False))
print("Pearson correlation was calculated using training data only.")


# 20. Feature Selection — Chi-Square

Chi-Square is appropriate for studying the relationship between categorical/discrete features and a categorical target.

\[
\chi^2 = \sum \frac{(O-E)^2}{E}
\]

\[
E=\frac{RowTotal\times ColumnTotal}{GrandTotal}
\]

\[
df=(r-1)(c-1)
\]

We manually construct contingency tables and calculate expected frequencies and contributions.

In [ ]:
categorical_candidates = [
    c for c in ["workclass","education","marital-status","occupation",
                "relationship","race","sex","native-country"]
    if c in train_df.columns
]

chi_results = []
for col in categorical_candidates:
    table = contingency_table(train_df[col], train_df["target"])
    chi2, df_chi, expected, contributions = manual_chi_square(table)
    chi_results.append([col, chi2, df_chi])

chi_table = pd.DataFrame(chi_results, columns=["Feature","Chi-Square","df"])
display(chi_table.sort_values("Chi-Square", ascending=False))

# Show one complete manual example
example_feature = categorical_candidates[0]
example_table = contingency_table(train_df[example_feature], train_df["target"])
chi2, df_chi, expected, contributions = manual_chi_square(example_table)
print("Example feature:", example_feature)
print("Observed:")
display(example_table)
print("Expected:")
display(expected)
print("Contributions:")
display(contributions)
print("Total Chi-Square:", chi2, "Degrees of freedom:", df_chi)
print("Chi-Square calculations use training data only.")


# 21. Feature Selection — ANOVA F-Test

ANOVA compares a numerical feature across categorical target groups.

We calculate:
- overall mean,
- group means,
- between-group variation,
- within-group variation,
- degrees of freedom,
- F-statistic.

A relatively high F-statistic indicates that group means differ substantially relative to within-group variation.

In [ ]:
anova_results = []
for col in train_df.select_dtypes(include=np.number).columns:
    if col == "target":
        continue
    groups = [
        train_df.loc[train_df["target"] == 0, col].dropna().to_numpy(),
        train_df.loc[train_df["target"] == 1, col].dropna().to_numpy()
    ]
    result = manual_anova(groups)
    anova_results.append([col, result["f_statistic"], result["grand_mean"]])

anova_table = pd.DataFrame(anova_results, columns=["Feature","F-statistic","Grand Mean"])
display(anova_table.sort_values("F-statistic", ascending=False))
print("ANOVA was calculated using training data only.")


# 22. Feature Selection — Mutual Information

For discrete/categorical variables:

\[
H(Y)=-\sum P(y)\log_2 P(y)
\]

\[
MI(X;Y)=H(Y)-H(Y|X)
\]

Mutual information measures shared information and can capture more general statistical dependence than Pearson correlation.

For this assignment, the manual calculation is demonstrated on categorical/discrete data rather than pretending an arbitrary frequency table is an exact estimator for continuous variables.

In [ ]:
mi_results = []

for col in categorical_candidates:
    mi = manual_mutual_information(train_df[col], train_df["target"])
    mi_results.append([col, mi])

mi_table = pd.DataFrame(mi_results, columns=["Feature","Manual MI (bits)"])
display(mi_table.sort_values("Manual MI (bits)", ascending=False))

print("Target entropy H(Y):", entropy(train_df["target"]))
mi_demo_feature = categorical_candidates[0]
print("Manual MI for", mi_demo_feature, ":", manual_mutual_information(train_df[mi_demo_feature], train_df["target"]))
print("Mutual Information was calculated using training data only.")


# 23. Feature-Selection Method Comparison

| Method | Feature Type | Target Type | Main Purpose |
|---|---|---|---|
| Variance | Numerical | Not required | Detect low-variance features |
| Pearson | Numerical | Numerical/binary encoded | Linear relationship |
| Chi-Square | Categorical/Discrete | Categorical | Statistical dependence |
| ANOVA F-Test | Numerical | Categorical | Difference across groups |
| Mutual Information | Discrete/Categorical for manual calculation | Discrete/Categorical | Shared information/dependence |

**Important:** We do not apply every technique blindly. The technique is chosen according to the data type and the question being studied.

# 24. Final Feature Selection Decision

The final decision is based on training-set evidence only. Numerical features are ranked using the combined ranks of absolute Pearson correlation and ANOVA F-statistic. Categorical features are ranked using the combined ranks of Chi-Square and Mutual Information.

To keep the process reproducible and avoid arbitrary manual choices, the strongest 4 numerical and strongest 5 categorical features are retained. The decision table below records the evidence and justification for every feature.


In [ ]:
# Reproducible final feature decision based only on training-set evidence.
# We retain the strongest 4 numerical and strongest 5 categorical features by combined ranks.
# This reduces dimensionality while preserving features with stronger statistical evidence.

num_features = variance_table["Feature"].tolist()
num_evidence = variance_table[["Feature","Manual Variance"]].merge(
    pearson_table[["Feature","Pearson r"]], on="Feature"
).merge(anova_table[["Feature","F-statistic"]], on="Feature")
num_evidence["Pearson_rank"] = num_evidence["Pearson r"].abs().rank(method="min", ascending=False)
num_evidence["ANOVA_rank"] = num_evidence["F-statistic"].rank(method="min", ascending=False)
num_evidence["Combined_rank"] = num_evidence["Pearson_rank"] + num_evidence["ANOVA_rank"]
num_evidence = num_evidence.sort_values(["Combined_rank","Feature"])

cat_evidence = chi_table[["Feature","Chi-Square"]].merge(
    mi_table[["Feature","Manual MI (bits)" ]], on="Feature"
)
cat_evidence["Chi_rank"] = cat_evidence["Chi-Square"].rank(method="min", ascending=False)
cat_evidence["MI_rank"] = cat_evidence["Manual MI (bits)"].rank(method="min", ascending=False)
cat_evidence["Combined_rank"] = cat_evidence["Chi_rank"] + cat_evidence["MI_rank"]
cat_evidence = cat_evidence.sort_values(["Combined_rank","Feature"])

keep_num = num_evidence.head(min(4, len(num_evidence)))["Feature"].tolist()
keep_cat = cat_evidence.head(min(5, len(cat_evidence)))["Feature"].tolist()
selected_features = keep_num + keep_cat

rows = []
for _, r in num_evidence.iterrows():
    keep = r["Feature"] in keep_num
    rows.append([r["Feature"], "Numerical", f"Pearson |r|={abs(r['Pearson r']):.4f}; ANOVA F={r['F-statistic']:.2f}", "Keep" if keep else "Remove",
                 "Strong combined training-set evidence" if keep else "Lower combined rank than retained numerical features"])
for _, r in cat_evidence.iterrows():
    keep = r["Feature"] in keep_cat
    rows.append([r["Feature"], "Categorical", f"Chi-square={r['Chi-Square']:.2f}; MI={r['Manual MI (bits)']:.4f}", "Keep" if keep else "Remove",
                 "Strong combined training-set evidence" if keep else "Lower combined rank than retained categorical features"])

decision_table = pd.DataFrame(rows, columns=["Feature","Type","Evidence","Decision","Justification"])
display(decision_table)

print("Final selected features:")
print(selected_features)
print("Number of original modelling features:", len(model_df.columns)-1)
print("Number of selected features:", len(selected_features))


# 25. Before vs After Summary

The completed Before vs After table is generated automatically in the code cell below using the actual dataset and final selected features.

The table compares data quality, record count and feature dimensionality before and after the workflow.


In [ ]:

# Final Before vs After summary
original_feature_count = len(df.columns) - 1
selected_feature_count = len(selected_features)
original_categorical_count = len([c for c in df.columns if df[c].dtype == "object" and c != "income"])
selected_categorical_count = sum(1 for c in selected_features if c in categorical_candidates)

before_after = pd.DataFrame({
    "Parameter": [
        "Records", "Features", "Missing Values", "Duplicate Records",
        "Categorical Features", "Potential Outliers (capital-gain, IQR)", "Selected Features"
    ],
    "Before": [
        len(df), original_feature_count, int(df.isna().sum().sum()), int(duplicate_count),
        original_categorical_count, len(iqr_outliers), original_feature_count
    ],
    "After": [
        len(model_df), selected_feature_count, int(model_df.isna().sum().sum()), 0,
        selected_categorical_count, len(iqr_outliers), selected_feature_count
    ]
})
display(before_after)

print("Interpretation:")
print(f"Records changed from {len(df)} to {len(model_df)} after duplicate removal.")
print(f"Modelling features were reduced from {original_feature_count} to {selected_feature_count}.")
print(f"Categorical modelling features represented in the final selection: {selected_categorical_count}.")
print("Potential outliers were identified but retained because mathematical outliers are not automatically errors.")
print("The supervised feature-selection statistics were calculated from the training set only.")


# 26. Library Verification

After the from-scratch calculations, library implementations may be used to verify selected results.

The goal is not to replace the manual implementation. It is to demonstrate that the underlying logic agrees with established implementations.

In [ ]:
# Verification examples
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import chi2, mutual_info_classif
from scipy.stats import pearsonr, f_oneway

# Min-Max verification
scaler = MinMaxScaler()
lib_norm = scaler.fit_transform(clean_df[["age"]]).ravel()
print("Max absolute difference, manual vs MinMaxScaler:",
      np.max(np.abs(np.array(manual_norm) - lib_norm)))

# Standardization verification
std_scaler = StandardScaler()
lib_z = std_scaler.fit_transform(clean_df[["age"]]).ravel()
print("Max absolute difference, manual vs StandardScaler:",
      np.max(np.abs(np.array(standardized) - lib_z)))

# Pearson verification
r_manual = manual_pearson(model_df["age"], model_df["target"])
r_library, p_library = pearsonr(model_df["age"], model_df["target"])
print("Manual Pearson:", r_manual)
print("Library Pearson:", r_library)
print("p-value:", p_library)

# ANOVA verification
g0 = model_df.loc[model_df["target"]==0, "age"].dropna()
g1 = model_df.loc[model_df["target"]==1, "age"].dropna()
f_library, p_anova = f_oneway(g0, g1)
print("Manual ANOVA F:", manual_anova([g0.to_numpy(), g1.to_numpy()])["f_statistic"])
print("Library ANOVA F:", f_library)
print("ANOVA p-value:", p_anova)

# Chi-Square verification on one categorical feature
from scipy.stats import chi2_contingency
ct = pd.crosstab(train_df["workclass"], train_df["target"])
chi_lib, p_chi, dof_lib, expected_lib = chi2_contingency(ct, correction=False)
chi_manual, dof_manual, _, _ = manual_chi_square(ct)
print("Manual Chi-Square:", chi_manual)
print("Library Chi-Square:", chi_lib)
print("Chi-Square p-value:", p_chi)

# Mutual Information verification on the same categorical feature
from sklearn.metrics import mutual_info_score
mi_lib = mutual_info_score(train_df["workclass"], train_df["target"])
mi_manual = manual_mutual_information(train_df["workclass"], train_df["target"])
print("Manual MI (bits):", mi_manual)
print("Library MI (nats):", mi_lib)
print("Note: the values use different logarithm bases, so direct numerical equality is not expected.")


# 27. Optional Model-Based Validation

This section is optional according to the assignment.

If used, compare:
- model with all suitable features,
- model with selected features.

Do not focus only on accuracy. Discuss dimensionality, redundancy, interpretability, training time and whether performance was maintained or improved.

In [ ]:
# Optional extension:
# Build a simple Logistic Regression model after the final feature-selection
# decisions have been reviewed by the group.
#
# Keep this section after the preprocessing/feature-selection work so that
# the assignment remains focused on understanding preprocessing and selection.

# 27A. Export Final Results

The following cell saves the important tables and graphs into the repository's `results/outputs` and `results/graphs` folders so they can be included in the GitHub submission.


In [ ]:

# Export final tables and graphs for the GitHub repository.
GRAPH_DIR = "../results/graphs"
OUTPUT_DIR = "../results/outputs"
os.makedirs(GRAPH_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

variance_table.to_csv(os.path.join(OUTPUT_DIR, "variance_results.csv"), index=False)
pearson_table.to_csv(os.path.join(OUTPUT_DIR, "pearson_results.csv"), index=False)
chi_table.to_csv(os.path.join(OUTPUT_DIR, "chi_square_results.csv"), index=False)
anova_table.to_csv(os.path.join(OUTPUT_DIR, "anova_results.csv"), index=False)
mi_table.to_csv(os.path.join(OUTPUT_DIR, "mutual_information_results.csv"), index=False)
decision_table.to_csv(os.path.join(OUTPUT_DIR, "final_feature_decision.csv"), index=False)
before_after.to_csv(os.path.join(OUTPUT_DIR, "before_after_summary.csv"), index=False)

# Save five required visualization types.
plt.figure(figsize=(8,5)); plt.hist(clean_df["age"], bins=30); plt.title("Age Distribution"); plt.xlabel("Age"); plt.ylabel("Frequency"); plt.tight_layout(); plt.savefig(os.path.join(GRAPH_DIR,"01_age_histogram.png"), dpi=150); plt.show()
plt.figure(figsize=(8,5)); plt.boxplot(clean_df["hours-per-week"]); plt.title("Hours Per Week - Box Plot"); plt.ylabel("Hours Per Week"); plt.tight_layout(); plt.savefig(os.path.join(GRAPH_DIR,"02_hours_boxplot.png"), dpi=150); plt.show()
income_counts = clean_df["income"].value_counts(); plt.figure(figsize=(7,5)); plt.bar(income_counts.index.astype(str), income_counts.values); plt.title("Income Class Distribution"); plt.xlabel("Income Class"); plt.ylabel("Number of Records"); plt.tight_layout(); plt.savefig(os.path.join(GRAPH_DIR,"03_income_bar.png"), dpi=150); plt.show()
plt.figure(figsize=(8,5)); plt.scatter(clean_df["age"], clean_df["hours-per-week"], alpha=0.2); plt.title("Age vs Hours Per Week"); plt.xlabel("Age"); plt.ylabel("Hours Per Week"); plt.tight_layout(); plt.savefig(os.path.join(GRAPH_DIR,"04_age_hours_scatter.png"), dpi=150); plt.show()
numeric_df = clean_df.select_dtypes(include=np.number); corr = numeric_df.corr(); plt.figure(figsize=(9,7)); plt.imshow(corr, aspect="auto"); plt.colorbar(label="Correlation"); plt.xticks(range(len(corr.columns)), corr.columns, rotation=90); plt.yticks(range(len(corr.columns)), corr.columns); plt.title("Numerical Feature Correlation Heatmap"); plt.tight_layout(); plt.savefig(os.path.join(GRAPH_DIR,"05_correlation_heatmap.png"), dpi=150); plt.show()

with open(os.path.join(OUTPUT_DIR, "final_selected_features.txt"), "w", encoding="utf-8") as f:
    f.write("Final selected features\n")
    f.write("========================\n")
    for feature in selected_features:
        f.write(f"- {feature}\n")

print("Export complete. Results saved to ../results/outputs and ../results/graphs")


# 28. Final Conclusions

The code cell below prints the final selected features and concise findings using the actual results from this run.


In [ ]:

print("KEY FINDINGS")
print("============")
print(f"1. Original dataset: {len(df)} records and {len(df.columns)-1} modelling features.")
print(f"2. Duplicate rows detected and removed: {duplicate_count}.")
print(f"3. Missing values remaining after the final modelling workflow: {int(model_df.isna().sum().sum())}.")
print(f"4. IQR detected {len(iqr_outliers)} potential capital-gain outliers; they were retained pending domain validation.")
print("5. Log transformation was investigated for the highly skewed capital-gain variable.")
print("6. Manual Min-Max, Standardization, Pearson and ANOVA calculations matched library verification results within floating-point precision.")
print("7. Variance, Pearson, Chi-Square, ANOVA and Mutual Information feature-selection calculations used training-set evidence only.")
print(f"8. Final selection reduced the modelling feature count from {len(model_df.columns)-1} to {len(selected_features)}.")

print("\nFINAL SELECTED FEATURES")
print("========================")
for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

print("\nFINAL REMARK")
print("============")
print("The project follows Understand → Calculate → Code → Verify → Interpret, with supervised decisions learned from training data to reduce data leakage.")
